# Mid-stage Evaluate All
Eval sach toan bo checkpoints local. Khong dung cached metrics.

In [1]:
import os, sys, gc, json
import torch
import torch.nn as nn
import numpy as np

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT_DIR)

# Override paths truoc khi import train_common
import train_common
train_common.BASE_DIR       = ROOT_DIR
train_common.RGBT_DATA_DIR  = os.path.join(ROOT_DIR, 'RGB-T', 'data', 'RGBTDronePerson')
train_common.LLVIP_DIR      = os.path.join(ROOT_DIR, 'RGB-T', 'data', 'LLVIP')
train_common.BACKBONES_DIR  = os.path.join(ROOT_DIR, 'backbones')

from train_common import *

MID_OUT   = os.path.join(ROOT_DIR, 'Mid-stage', 'outputs')
BB_DIR    = os.path.join(ROOT_DIR, 'backbones')
RGBT_DIR  = train_common.RGBT_DATA_DIR

RGB_S1 = os.path.join(BB_DIR, 'SDS_RGB_best.pt')
RGB_S2 = os.path.join(BB_DIR, 'llvip_rgb_best.pt')
THR    = os.path.join(BB_DIR, 'llvip_thermal_best.pt')

SEEDS      = [42, 777, 123]
IMG_SIZE   = 640
BATCH_SIZE = 8

print(f'ROOT_DIR:  {ROOT_DIR}')
print(f'RGBT_DIR:  {RGBT_DIR} | exists={os.path.exists(RGBT_DIR)}')
print(f'Device:    {device}')

# Setup fusion_yolo dataset (copy anh vao dung dinh dang YOLO)
FUSION_DIR = setup_mid_dataset(os.path.join(ROOT_DIR, 'data'))
print(f'Fusion dir: {FUSION_DIR}')

val_ds     = RGBTDataset(FUSION_DIR, 'val', IMG_SIZE)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate_fn, pin_memory=False)
print(f'Val samples: {len(val_ds)}')


ROOT_DIR:  e:\FPTU\AIP491
RGBT_DIR:  e:\FPTU\AIP491\RGB-T\data\RGBTDronePerson | exists=True
Device:    cuda
Dataset da ton tai: e:\FPTU\AIP491\data\fusion_yolo (4760 labels)
Fusion dir: e:\FPTU\AIP491\data\fusion_yolo
RGBTDataset [val]: 1207 mau | augment: none
Val samples: 1207


In [2]:
import torch.nn.functional as F
import random
from ultralytics.nn.modules import C2f, Conv, Detect

# ── Baseline Fusion ──────────────────────────────────────────
class BaselineFusion(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}
    def __init__(self, rgb_bb, thr_bb, nc=1):
        super().__init__()
        self.rgb_stream = rgb_bb
        self.thermal_stream = thr_bb
        self.fuse_p3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU())
        self.fuse_p4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU())
        self.fuse_p5 = nn.Sequential(nn.Conv2d(512, 512, 1, bias=False), nn.BatchNorm2d(512), nn.SiLU())
        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(768, 256, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(384, 128, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(128, 128, 3, 2)
        self.bu_c2f_p4  = C2f(384, 256, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(256, 256, 3, 2)
        self.bu_c2f_p5  = C2f(768, 512, n=1, shortcut=False)
        self.detect     = Detect(nc=nc, ch=(128, 256, 512))
        self.detect.stride = torch.tensor([8., 16., 32.])
    def _extract(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS: feats[i] = x
        return feats
    def forward(self, rgb, thr):
        rf = self._extract(self.rgb_stream, rgb)
        tf = self._extract(self.thermal_stream, thr)
        p3 = self.fuse_p3(torch.cat([rf[4], tf[4]], 1))
        p4 = self.fuse_p4(torch.cat([rf[6], tf[6]], 1))
        p5 = self.fuse_p5(torch.cat([rf[9], tf[9]], 1))
        p4t = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], 1))
        p3o = self.td_c2f_p3(torch.cat([self.upsample(p4t), p3], 1))
        p4o = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3o), p4t], 1))
        p5o = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4o), p5], 1))
        return self.detect([p3o, p4o, p5o])

# ── Progressive Fusion ───────────────────────────────────────
class ProgressiveFusion(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}
    def __init__(self, rgb_bb, thr_bb, nc=1):
        super().__init__()
        self.rgb_stream = rgb_bb
        self.thermal_stream = thr_bb
        self.fuse_p3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), nn.BatchNorm2d(128), nn.SiLU())
        self.fuse_p4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), nn.BatchNorm2d(256), nn.SiLU())
        self.fuse_p5 = nn.Sequential(nn.Conv2d(512, 512, 1, bias=False), nn.BatchNorm2d(512), nn.SiLU())
        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(512 + 256, 256, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(128, 128, 3, 2)
        self.bu_c2f_p4  = C2f(128 + 256, 256, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(256, 256, 3, 2)
        self.bu_c2f_p5  = C2f(256 + 512, 512, n=1, shortcut=False)
        self.detect     = Detect(nc=nc, ch=(128, 256, 512))
        self.detect.stride = torch.tensor([8., 16., 32.])
    def _extract(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS: feats[i] = x
        return feats
    def forward(self, rgb, thr):
        rf = self._extract(self.rgb_stream, rgb)
        tf = self._extract(self.thermal_stream, thr)
        p3 = self.fuse_p3(torch.cat([rf[4], tf[4]], 1))
        p4 = self.fuse_p4(torch.cat([rf[6], tf[6]], 1))
        p5 = self.fuse_p5(torch.cat([rf[9], tf[9]], 1))
        p4t = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], 1))
        p3o = self.td_c2f_p3(torch.cat([self.upsample(p4t), p3], 1))
        p4o = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3o), p4t], 1))
        p5o = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4o), p5], 1))
        return self.detect([p3o, p4o, p5o])

# ── QFDet helpers ────────────────────────────────────────────
class PoolUpsample(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.pool   = nn.MaxPool2d(kernel_size=2, stride=2)
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels), nn.SiLU()
        )
    def forward(self, feat):
        pooled    = self.pool(feat)
        upsampled = F.interpolate(pooled, size=feat.shape[2:], mode='bilinear', align_corners=False)
        return self.reduce(torch.cat([feat, upsampled], dim=1))

class QualityAwareFusion(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid = max(in_channels // 4, 16)
        self.q_rgb = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU(),
            nn.Conv2d(mid, 1, 1), nn.Sigmoid()
        )
        self.q_thr = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU(),
            nn.Conv2d(mid, 1, 1), nn.Sigmoid()
        )
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels), nn.SiLU()
        )
    def forward(self, rgb_feat, thr_feat):
        q_rgb = self.q_rgb(rgb_feat)
        q_thr = self.q_thr(thr_feat)
        w     = torch.softmax(torch.cat([q_rgb, q_thr], dim=1), dim=1)
        fused = (1 + w[:, 0:1]) * rgb_feat + (1 + w[:, 1:2]) * thr_feat
        return self.proj(fused)

# ── QFDet / ProgQFDet Fusion (dung chung 1 class) ───────────
class QFDetFusion(nn.Module):
    EXTRACT_LAYERS = {4: 64, 6: 128, 9: 256}
    def __init__(self, rgb_bb, thr_bb, nc=1):
        super().__init__()
        self.rgb_stream     = rgb_bb
        self.thermal_stream = thr_bb
        self.pool_rgb_p3 = PoolUpsample(64);  self.pool_rgb_p4 = PoolUpsample(128);  self.pool_rgb_p5 = PoolUpsample(256)
        self.pool_thr_p3 = PoolUpsample(64);  self.pool_thr_p4 = PoolUpsample(128);  self.pool_thr_p5 = PoolUpsample(256)
        self.fuse_p3 = QualityAwareFusion(64)
        self.fuse_p4 = QualityAwareFusion(128)
        self.fuse_p5 = QualityAwareFusion(256)
        self.upsample   = nn.Upsample(scale_factor=2, mode='nearest')
        self.td_c2f_p4  = C2f(256 + 128, 128, n=1, shortcut=False)
        self.td_c2f_p3  = C2f(128 + 64,   64, n=1, shortcut=False)
        self.bu_conv_p4 = Conv(64,  64,  3, 2)
        self.bu_c2f_p4  = C2f(64  + 128, 128, n=1, shortcut=False)
        self.bu_conv_p5 = Conv(128, 128, 3, 2)
        self.bu_c2f_p5  = C2f(128 + 256, 256, n=1, shortcut=False)
        self.detect = Detect(nc=nc, ch=(64, 128, 256))
        self.detect.stride = torch.tensor([8., 16., 32.])
    def _extract(self, stream, x):
        feats = {}
        for i, layer in enumerate(stream):
            x = layer(x)
            if i in self.EXTRACT_LAYERS: feats[i] = x
        return feats
    def forward(self, rgb, thr):
        rf = self._extract(self.rgb_stream, rgb)
        tf = self._extract(self.thermal_stream, thr)
        p3 = self.fuse_p3(self.pool_rgb_p3(rf[4]), self.pool_thr_p3(tf[4]))
        p4 = self.fuse_p4(self.pool_rgb_p4(rf[6]), self.pool_thr_p4(tf[6]))
        p5 = self.fuse_p5(self.pool_rgb_p5(rf[9]), self.pool_thr_p5(tf[9]))
        p4t = self.td_c2f_p4(torch.cat([self.upsample(p5), p4], dim=1))
        p3o = self.td_c2f_p3(torch.cat([self.upsample(p4t), p3], dim=1))
        p4o = self.bu_c2f_p4(torch.cat([self.bu_conv_p4(p3o), p4t], dim=1))
        p5o = self.bu_c2f_p5(torch.cat([self.bu_conv_p5(p4o), p5], dim=1))
        return self.detect([p3o, p4o, p5o])

print('Model classes OK: BaselineFusion, ProgressiveFusion, QFDetFusion')


Model classes OK: BaselineFusion, ProgressiveFusion, QFDetFusion


In [3]:
METHODS = [
    ('progressive_stream1', RGB_S1, THR, ProgressiveFusion),
    ('progressive_stream2', RGB_S2, THR, ProgressiveFusion),
    ('qfdet_stream1',       RGB_S1, THR, QFDetFusion),
    ('qfdet_stream2',       RGB_S2, THR, QFDetFusion),
    ('prog_qfdet_stream1',  RGB_S1, THR, QFDetFusion),
    ('prog_qfdet_stream2',  RGB_S2, THR, QFDetFusion),
]

all_results = {}

for method_name, rgb_path, thr_path, ModelClass in METHODS:
    method_dir = os.path.join(MID_OUT, method_name)
    if not os.path.exists(method_dir):
        print(f'[SKIP] {method_name}: not found')
        continue

    print(f'\n{"="*55}')
    print(f'  {method_name}')
    print(f'{"="*55}')
    method_results = {}

    for SEED in SEEDS:
        best_path = os.path.join(method_dir, f'seed_{SEED}', 'fusion_best.pt')
        if not os.path.exists(best_path):
            print(f'  Seed {SEED}: not found, skip')
            continue

        set_seed(SEED)
        rgb_bb = load_yolov8n_backbone(rgb_path)
        thr_bb = load_yolov8n_backbone(thr_path)
        model  = ModelClass(rgb_bb, thr_bb, nc=1).to(device)

        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])

        metrics = evaluate_model(model, val_loader, img_size=IMG_SIZE)
        method_results[SEED] = metrics
        print(f'  Seed {SEED}:  P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}  '
              f'mAP@0.5={metrics["map50"]:.4f}  mAP@.5:.95={metrics["map50_95"]:.4f}')

        del model, rgb_bb, thr_bb
        torch.cuda.empty_cache(); gc.collect()

    if method_results:
        maps   = [v['map50']     for v in method_results.values()]
        m5095  = [v['map50_95']  for v in method_results.values()]
        print(f'  --> Mean mAP@0.5={sum(maps)/len(maps):.4f}  mAP@.5:.95={sum(m5095)/len(m5095):.4f}')
        all_results[method_name] = method_results

with open(os.path.join(MID_OUT, 'eval_summary.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved: {MID_OUT}/eval_summary.json')



  progressive_stream1
  Seed 42:  P=0.6740  R=0.5513  mAP@0.5=0.5880  mAP@.5:.95=0.1989
  Seed 777:  P=0.6742  R=0.5325  mAP@0.5=0.5689  mAP@.5:.95=0.1900
  Seed 123:  P=0.6257  R=0.5364  mAP@0.5=0.5441  mAP@.5:.95=0.1770
  --> Mean mAP@0.5=0.5670  mAP@.5:.95=0.1886

  progressive_stream2
  Seed 42:  P=0.6970  R=0.5402  mAP@0.5=0.5870  mAP@.5:.95=0.1968
  Seed 777:  P=0.6822  R=0.5442  mAP@0.5=0.5904  mAP@.5:.95=0.2004
  Seed 123:  P=0.6033  R=0.6047  mAP@0.5=0.5543  mAP@.5:.95=0.1892
  --> Mean mAP@0.5=0.5772  mAP@.5:.95=0.1955

  qfdet_stream1
  Seed 42:  P=0.6864  R=0.4114  mAP@0.5=0.5256  mAP@.5:.95=0.1725
  Seed 777:  P=0.6060  R=0.4847  mAP@0.5=0.5001  mAP@.5:.95=0.1662
  Seed 123:  P=0.6199  R=0.4938  mAP@0.5=0.5066  mAP@.5:.95=0.1560
  --> Mean mAP@0.5=0.5108  mAP@.5:.95=0.1649

  qfdet_stream2
  Seed 42:  P=0.6241  R=0.5245  mAP@0.5=0.5288  mAP@.5:.95=0.1781
  Seed 777:  P=0.6423  R=0.4331  mAP@0.5=0.5033  mAP@.5:.95=0.1642
  Seed 123:  P=0.5934  R=0.5544  mAP@0.5=0.5238  mAP

In [4]:
print(f'{"Method":<30} {"S42":>8} {"S777":>8} {"S123":>8} {"Mean":>8}')
print('-' * 60)
for method, results in sorted(all_results.items(), key=lambda x: -sum(v['map50'] for v in x[1].values())/len(x[1])):
    vals = {s: results[s]['map50'] for s in SEEDS if s in results}
    mean = sum(vals.values()) / len(vals)
    cols = ''.join(f'{vals.get(s, 0):8.4f}' for s in SEEDS)
    print(f'{method:<30}{cols}{mean:8.4f}')

Method                              S42     S777     S123     Mean
------------------------------------------------------------
progressive_stream2             0.5870  0.5904  0.5543  0.5772
progressive_stream1             0.5880  0.5689  0.5441  0.5670
prog_qfdet_stream1              0.5432  0.5558  0.5797  0.5596
prog_qfdet_stream2              0.5374  0.5716  0.5501  0.5530
qfdet_stream2                   0.5288  0.5033  0.5238  0.5186
qfdet_stream1                   0.5256  0.5001  0.5066  0.5108
